# U.S. 2024 Presidential Election TikTok dataset

This notebook analyzes only the U.S. election dataset. It checks local availability, public structure, release sizes, ID quality, overlap, paper-count agreement, reconstruction limits, and collection method. All findings are collected at the end.

## Sources and local files

- Paper: [Tracking the 2024 U.S. Presidential Election Chatter on TikTok](https://arxiv.org/abs/2412.15583)
- Official repository: [US2024PresElectionTikToks](https://github.com/gabbypinto/US2024PresElectionTikToks)
- Local folder: `downloaded_datasets/us_election/`

The three CSVs are overlapping releases of one project, not three independent datasets. Their local byte sizes and Git blob hashes were checked against the official GitHub repository on 2026-09-17.

In [1]:
from pathlib import Path
import hashlib
import re
import pandas as pd
from IPython.display import display

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
ROOT = Path.cwd()
if not (ROOT / 'downloaded_datasets').exists() and (ROOT.parent / 'downloaded_datasets').exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'downloaded_datasets' / 'us_election'
FILES = {
    'Memo1': DATA_DIR / 'TikTok_IDs_Memo1.csv',
    'v2': DATA_DIR / 'TikTok_IDs_v2.csv',
    'Feb 2025': DATA_DIR / 'TikTok_IDs_2025_Feb.csv',
}
missing = [str(p) for p in FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError('Missing local files:\n' + '\n'.join(missing))
print('Local data directory:', DATA_DIR)

Local data directory: /Users/wang/Documents/tiktok/downloaded_datasets/us_election


## 1. Verify the local official files

GitHub identifies each file with a Git blob SHA-1. Matching this value verifies the complete local file, not only its filename or row count.

In [2]:
OFFICIAL = {
    'Memo1': {'bytes': 35_986_663, 'git_blob': '0afc4e6eec400a2fc0e0fe2598133694ae6e6dc1'},
    'v2': {'bytes': 93_841_027, 'git_blob': '5f9c21d8551ccfd29172072dcb8acbc9506fea59'},
    'Feb 2025': {'bytes': 81_398_163, 'git_blob': '2ae4bf092826ce3dea9bb08c3fbd98673d9b1289'},
}

def git_blob_sha1(path):
    size = path.stat().st_size
    digest = hashlib.sha1(f'blob {size}\0'.encode())
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

availability = []
for release, path in FILES.items():
    actual_blob = git_blob_sha1(path)
    availability.append({
        'release': release, 'local_file': path.name,
        'size_MiB': path.stat().st_size / 1024**2,
        'bytes_match_official': path.stat().st_size == OFFICIAL[release]['bytes'],
        'git_blob_matches_official': actual_blob == OFFICIAL[release]['git_blob'],
    })
display(pd.DataFrame(availability).round({'size_MiB': 2}))

,release,local_file,size_MiB,bytes_match_official,git_blob_matches_official
0,Memo1,TikTok_IDs_Memo1.csv,34.32,True,True
1,v2,TikTok_IDs_v2.csv,89.49,True,True
2,Feb 2025,TikTok_IDs_2025_Feb.csv,77.63,True,True


## 2. Structure and ID quality

The public release is intentionally smaller than the internal research corpus. It releases IDs, not raw video or the full metadata used in the paper.

In [3]:
for release, path in FILES.items():
    preview = pd.read_csv(path, nrows=3, dtype=str)
    print(f'\n{release}: {preview.shape[1]} columns -> {list(preview.columns)}')
    display(preview)


Memo1: 1 columns -> ['id']


,id
0,7298132049062382891
1,7298132047581924654
2,7298131955592465710



v2: 2 columns -> ['Unnamed: 0', 'id']


,Unnamed: 0,id
0,0,7.297389872858566e+18
1,1,7.297389853237317e+18
2,2,7.297389825458441e+18



Feb 2025: 1 columns -> ['id']


,id
0,7409828458894789931
1,7409828193252756779
2,7409827326789356846


In [4]:
EXACT_ID = re.compile(r'^\d{19}$')
SCIENTIFIC_ID = re.compile(r'^\d(?:\.\d+)?e\+18$', re.IGNORECASE)

def profile_ids(path, chunksize=500_000):
    rows = missing = exact = scientific = 0
    unique = set()
    for chunk in pd.read_csv(path, usecols=['id'], dtype=str, chunksize=chunksize):
        values = chunk['id']
        rows += len(values)
        missing += int(values.isna().sum())
        clean = values.dropna().str.strip()
        exact += int(clean.str.fullmatch(EXACT_ID).sum())
        scientific += int(clean.str.fullmatch(SCIENTIFIC_ID).sum())
        unique.update(clean.tolist())
    return {
        'rows': rows, 'unique_text_values': len(unique),
        'duplicate_rows': rows - missing - len(unique),
        'missing': missing, 'exact_19_digit_IDs': exact,
        'scientific_notation_IDs': scientific,
        'other_invalid_values': rows - missing - exact - scientific,
    }

profiles = pd.DataFrame({release: profile_ids(path) for release, path in FILES.items()}).T
display(profiles)

,rows,unique_text_values,duplicate_rows,missing,exact_19_digit_IDs,scientific_notation_IDs,other_invalid_values
Memo1,1799333,1799333,0,0,1799333,0,0
v2,3176949,3176949,0,0,0,3176948,1
Feb 2025,4069908,4069908,0,0,4069908,0,0


## 3. Interesting point: paper and release counts changed over time

In [5]:
PAPER_VIDEOS = 3_144_836
counts = pd.DataFrame([
    {'snapshot': 'Paper', 'coverage': 'paper snapshot through 16 Oct 2024', 'videos_or_IDs': PAPER_VIDEOS},
    {'snapshot': 'Repository v2', 'coverage': 'posts 1 Nov 2023 to 24 Oct 2024', 'videos_or_IDs': int(profiles.loc['v2', 'rows'])},
    {'snapshot': 'Repository Feb 2025', 'coverage': 'posts 1 Nov 2023 to 20 Jan 2025', 'videos_or_IDs': int(profiles.loc['Feb 2025', 'rows'])},
])
counts['difference_from_paper'] = counts['videos_or_IDs'] - PAPER_VIDEOS
counts['percent_difference'] = counts['difference_from_paper'] / PAPER_VIDEOS * 100
display(counts.round({'percent_difference': 2}))

,snapshot,coverage,videos_or_IDs,difference_from_paper,percent_difference
0,Paper,paper snapshot through 16 Oct 2024,3144836,0,0.00
1,Repository v2,posts 1 Nov 2023 to 24 Oct 2024,3176949,32113,1.02
2,Repository Feb 2025,posts 1 Nov 2023 to 20 Jan 2025,4069908,925072,29.42


## 4. Interesting point: the release files overlap

This check keeps the smaller Memo1 set in memory and streams through the February file.

In [6]:
def load_id_set(path, chunksize=500_000):
    values = set()
    for chunk in pd.read_csv(path, usecols=['id'], dtype=str, chunksize=chunksize):
        values.update(chunk['id'].dropna().str.strip())
    return values

memo_ids = load_id_set(FILES['Memo1'])
overlap = 0
for chunk in pd.read_csv(FILES['Feb 2025'], usecols=['id'], dtype=str, chunksize=500_000):
    overlap += int(chunk['id'].dropna().str.strip().isin(memo_ids).sum())

overlap_result = pd.DataFrame([{
    'Memo1 unique IDs': len(memo_ids),
    'also in Feb 2025': overlap,
    'absent from Feb 2025': len(memo_ids) - overlap,
    'overlap_percent': overlap / len(memo_ids) * 100,
}])
display(overlap_result.round({'overlap_percent': 2}))
del memo_ids

,Memo1 unique IDs,also in Feb 2025,absent from Feb 2025,overlap_percent
0,1799333,1294289,505044,71.93


## 5. Interesting point: v2 cannot preserve exact TikTok IDs

In [7]:
v2_sample = pd.read_csv(FILES['v2'], dtype=str, nrows=8)
display(pd.DataFrame({
    'stored_v2_value': v2_sample['id'],
    'unsafe_float_conversion': v2_sample['id'].map(lambda value: str(int(float(value)))),
}))
print('The scientific notation stores too few significant digits for a 19-digit TikTok ID.')

,stored_v2_value,unsafe_float_conversion
0,7.297389872858566e+18,7297389872858565632
1,7.297389853237317e+18,7297389853237316608
2,7.297389825458441e+18,7297389825458441216
3,7.297389728679202e+18,7297389728679201792
4,7.297389700514319e+18,7297389700514319360
5,7.297389651285888e+18,7297389651285888000
6,7.297389634219265e+18,7297389634219265024
7,7.297389628678524e+18,7297389628678523904


The scientific notation stores too few significant digits for a 19-digit TikTok ID.


## 6. Availability, reconstruction, and collection method

**Availability:** The ID CSVs and collection scripts are publicly downloadable from GitHub. The repository does not state a dataset licence. Separate Whisper transcripts are linked through Google Drive.

**Public structure:** The downloaded CSVs contain only IDs. They do not contain raw video, audio, captions, transcripts, OCR, usernames, descriptions, engagement values, comments, or annotations. The internal paper corpus had richer metadata, but it is not present in these public CSVs.

**Reconstruction:** The exact IDs in Memo1 and the February release can be used for API hydration. However, deleted/private videos cannot be recovered. The repository video-download helper expects a username as well as an ID, while these CSVs do not include usernames. The v2 scientific-notation IDs should not be used for exact reconstruction.

**Collection:** The authors used the official TikTok Research API. Queries selected U.S.-region videos through election keywords and hashtags, with date ranges organized into political-event phases. API responses requested ID, description, creation time, region, engagement counts, music ID, hashtags, username, and voice-to-text. The repository later extended publication coverage from 1 November 2023 to 20 January 2025. The exact wall-clock dates when every API request ran are not reported.

# Findings

1. The official files are genuinely stored locally. All three local files match the official GitHub byte counts and Git blob hashes.
2. This public download is an ID release, not the complete research dataset described by the paper. It contains no raw videos and no public per-video metadata.
3. The February 2025 file is the best release for exact identifiers: 4,069,908 rows, 4,069,908 unique 19-digit IDs, and no duplicates.
4. The three files are versions of one project and must not be added together. Memo1 has 1,799,333 IDs; 1,294,289 appear in the February file and 505,044 do not. Its overlap rate is 71.93%.
5. The paper count and repository counts are different because they represent different snapshots. v2 is 32,113 IDs above the paper count; the February release is 925,072 above it.
6. The v2 file has 3,176,948 IDs in scientific notation and one other invalid value. Exact trailing digits have been lost, so v2 is unsafe for reconstructing videos.
7. Exact IDs allow attempted hydration, not guaranteed recovery. Link decay, privacy changes, removal, regional access, and missing usernames still limit reproducibility.
8. The paper reports 693,826 unique users and 485,343 transcripts, but the public ID CSVs cannot independently verify those values.